# AXUM — Ge'ez OCR training (CNN + BiLSTM + CTC)

Retrain the artefact inscription OCR model on the **merged** HHD-Ethiopic + Yaredoffice dataset.

**Before this notebook (on your laptop):**
1. `python scripts/merge_datasets.py`
2. `python scripts/export_colab_training_data.py --data-root data/geez_merged --include-code`
3. Upload **both** zips from `exports/`:
   - `geez_merged_colab.zip` (images + CSV)
   - `geez_ocr_colab_code.zip` (fixed training code)

**Check the export is post-fix:** open `geez_merged_colab.manifest.json` on your laptop — `ocr_pipeline_fix_id` should be `ctc_filter_alignment_metrics_v1`.

**Runtime:** GPU recommended (Runtime → Change runtime type → T4). CPU works but is much slower.

**After training:** download `geez_ocr.pth` → place in `models/` on the rover laptop.

In [ ]:
# ── 1. GPU check ─────────────────────────────────────────────
import torch

USE_GPU = torch.cuda.is_available()
DEVICE = "cuda" if USE_GPU else "cpu"
print(f"Device: {DEVICE}")
if USE_GPU:
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Tip: Runtime → Change runtime type → T4 GPU for ~10× faster training")

In [ ]:
# ── 2. Install dependencies ──────────────────────────────────
%%capture
!pip install -q loguru tqdm albumentations opencv-python-headless pillow numpy scipy

In [ ]:
# ── 3. Upload archives (interactive) ─────────────────────────
from pathlib import Path
from google.colab import files

CONTENT = Path("/content")
DATA_ZIP = CONTENT / "geez_merged_colab.zip"
CODE_ZIP = CONTENT / "geez_ocr_colab_code.zip"
AXUM_ROOT = CONTENT / "axum"
DATA_ROOT = CONTENT / "data"

if not DATA_ZIP.exists():
    print("Upload geez_merged_colab.zip from exports/ on your laptop")
    uploaded = files.upload()
    DATA_ZIP = CONTENT / next(iter(uploaded))

if not CODE_ZIP.exists():
    print("Upload geez_ocr_colab_code.zip from exports/ on your laptop")
    uploaded = files.upload()
    CODE_ZIP = CONTENT / next(iter(uploaded))

!unzip -q -o "{DATA_ZIP}" -d "{DATA_ROOT}"
!unzip -q -o "{CODE_ZIP}" -d "{AXUM_ROOT}"

import sys
sys.path.insert(0, str(AXUM_ROOT))
print("AXUM root:", AXUM_ROOT)
print("Data root:", DATA_ROOT)

In [ ]:
# ── 4. Verify export has OCR fixes + CTC-safe labels ─────────
from src.ocr.pipeline import (
    HHDEthiopicDataset,
    label_fits_ctc,
    log_dataset_ctc_stats,
    min_ctc_timesteps_for_label,
)
from config import OCR_CTC_SEQ_LEN

DATA_DIR = DATA_ROOT / "geez_merged"
assert (DATA_DIR / "train_raw" / "image_text_pairs_train.csv").exists(), (
    f"Missing CSV under {DATA_DIR}/train_raw — re-export from laptop"
)

# Code fix markers (post 0%-bug pipeline)
pipeline_src = (AXUM_ROOT / "src/ocr/pipeline.py").read_text(encoding="utf-8")
fixes_ok = all(
    fn in pipeline_src
    for fn in ("label_fits_ctc", "compute_sequence_metrics", "OCR_CTC_SEQ_LEN")
)
print(f"OCR pipeline fixes present: {fixes_ok}")
print(f"CTC sequence budget: {OCR_CTC_SEQ_LEN} timesteps")
assert fixes_ok, "Re-export with --include-code after pulling latest AXUM"

train_ds = HHDEthiopicDataset(str(DATA_DIR), split="train", augment=False)
val_ds = HHDEthiopicDataset(str(DATA_DIR), split="val", augment=False)
print(f"Train samples: {len(train_ds):,} | Val: {len(val_ds):,}")

# Spot-check a few labels
sample_texts = [t for _, t in train_ds.samples[:5]]
for t in sample_texts:
    print(f"  label len={len(t):2d}  ctc_steps={min_ctc_timesteps_for_label(t):2d}  {t[:40]}")

In [ ]:
# ── 5. Training config (interactive sliders) ─────────────────
import ipywidgets as widgets
from IPython.display import display

epochs_slider = widgets.IntSlider(value=30, min=5, max=80, step=5, description="Epochs")
batch_slider = widgets.IntSlider(
    value=64 if USE_GPU else 32,
    min=8,
    max=128 if USE_GPU else 64,
    step=8,
    description="Batch",
)
lr_slider = widgets.FloatLogSlider(
    value=0.001,
    base=10,
    min=-4,
    max=-1,
    step=0.1,
    description="LR",
)

display(widgets.VBox([epochs_slider, batch_slider, lr_slider]))

EPOCHS = epochs_slider.value
BATCH_SIZE = batch_slider.value
LEARNING_RATE = lr_slider.value
SAVE_PATH = AXUM_ROOT / "models" / "geez_ocr.pth"
SAVE_PATH.parent.mkdir(parents=True, exist_ok=True)

print(f"Epochs={EPOCHS}  batch={BATCH_SIZE}  lr={LEARNING_RATE}  device={DEVICE}")
print(f"Checkpoint → {SAVE_PATH}")
print("Re-run the next cell after changing sliders.")

In [ ]:
# ── 6. Train ─────────────────────────────────────────────────
from scripts.train_ocr import create_weighted_sampler
from src.ocr.pipeline import train_ocr_model

result = train_ocr_model(
    data_dir=str(DATA_DIR),
    save_path=SAVE_PATH,
    num_epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    weighted_sampler_fn=create_weighted_sampler,
    device=DEVICE,
)

print("\nBest val loss:", result.get("best_val_loss"))
if result.get("epoch_metrics"):
    last = result["epoch_metrics"][-1]
    print(
        f"Last epoch — CharAcc: {last['char_accuracy']:.1%}, "
        f"Exact: {last.get('exact_match', 0):.1%}, "
        f"CER: {last.get('cer', 0):.3f}"
    )

In [ ]:
# ── 7. Quick sanity inference on a val sample ────────────────
import random
from src.ocr.model import load_ocr_model, IDX_TO_CHAR

model = load_ocr_model(SAVE_PATH)
idx = random.randint(0, len(val_ds) - 1)
tensor, label_tensor, _ = val_ds[idx]
true_text = "".join(IDX_TO_CHAR.get(i, "") for i in label_tensor.tolist())

with torch.no_grad():
    log_probs = model(tensor.unsqueeze(0).to(DEVICE))
    pred = model.decode_beam(log_probs)[0]

print("True: ", true_text)
print("Pred: ", pred)
print("Match:", pred == true_text)

## Download checkpoint to rover laptop

1. Run the cell below to download `geez_ocr.pth`.
2. Copy to `models/geez_ocr.pth` in your AXUM project.
3. Test: `python scripts/test_ocr_samples.py`

**Metrics guide:**
- **CharAcc** — alignment-based; should rise above 0% early on multi-char data
- **Exact** — full string match (stricter; expect low early)
- **CER** — character error rate; should trend down

In [ ]:
from google.colab import files

assert SAVE_PATH.exists(), "Train first — no checkpoint found"
files.download(str(SAVE_PATH))